# Interactive Training Workflow

Step-by-step guide to training the Siamese U-Net change detection model.
Use this notebook for experimentation and debugging before running
the full training script via CLI.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from src.config import Config
from src.data_loader import build_dataloaders
from src.models import build_model, build_loss, count_parameters
from src.train import train_one_epoch, validate, EarlyStopping
from src.utils import set_seed, get_device, visualize_change_detection

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

In [ ]:
# Load configuration
config = Config.from_yaml('../configs/default.yaml')

# Override for interactive use (smaller batch for quick iteration)
config.training.batch_size = 8
config.training.epochs = 5  # Quick test run

set_seed(config.seed)
device = get_device()
print(f'Device: {device}')
print(f'Config: {config.experiment_name}')

## Data Pipeline Verification

Before training, verify that the data pipeline produces correctly
shaped tensors and that augmentations look reasonable.

In [ ]:
# Build data loaders
loaders = build_dataloaders(
    data_root=config.paths.data_root,
    batch_size=config.training.batch_size,
    num_workers=0,  # Use 0 workers in notebooks to avoid multiprocessing issues
)

print(f'Splits available: {list(loaders.keys())}')
for split, loader in loaders.items():
    print(f'  {split}: {len(loader)} batches, {len(loader.dataset)} samples')

# Inspect a single batch
batch = next(iter(loaders['train']))
print(f'\nBatch shapes:')
print(f'  image1: {batch["image1"].shape}')
print(f'  image2: {batch["image2"].shape}')
print(f'  mask:   {batch["mask"].shape}')
print(f'  value range: [{batch["image1"].min():.2f}, {batch["image1"].max():.2f}]')

# Visualize a few samples from the batch
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for i in range(min(2, batch['image1'].shape[0])):
    img1 = batch['image1'][i].numpy()
    img2 = batch['image2'][i].numpy()
    mask = batch['mask'][i, 0].numpy()
    
    # Denormalize for display
    img1_disp = np.clip(img1.transpose(1,2,0) * [0.229,0.224,0.225] + [0.485,0.456,0.406], 0, 1)
    img2_disp = np.clip(img2.transpose(1,2,0) * [0.229,0.224,0.225] + [0.485,0.456,0.406], 0, 1)
    
    axes[i, 0].imshow(img1_disp)
    axes[i, 0].set_title('T1' if i == 0 else '')
    axes[i, 1].imshow(img2_disp)
    axes[i, 1].set_title('T2' if i == 0 else '')
    axes[i, 2].imshow(mask, cmap='hot')
    axes[i, 2].set_title('Mask' if i == 0 else '')
    for ax in axes[i]:
        ax.axis('off')
plt.tight_layout()
plt.show()

## Model Architecture

Inspect the Siamese U-Net architecture and verify parameter counts.

In [ ]:
# Build model
model = build_model(
    in_channels=config.model.in_channels,
    num_classes=config.model.num_classes,
    pretrained=config.model.pretrained,
    fusion_mode=config.model.fusion,
    deep_supervision=config.model.deep_supervision,
).to(device)

n_params = count_parameters(model)
print(f'Total trainable parameters: {n_params:,} ({n_params/1e6:.1f}M)')

# Verify forward pass with dummy data
dummy1 = torch.randn(2, 3, 256, 256).to(device)
dummy2 = torch.randn(2, 3, 256, 256).to(device)
model.train()
output = model(dummy1, dummy2)
print(f'\nOutput shape: {output["pred"].shape}')
if 'aux_preds' in output:
    print(f'Auxiliary outputs: {len(output["aux_preds"])} scales')
    for i, aux in enumerate(output['aux_preds']):
        print(f'  Scale {i}: {aux.shape}')

## Training

Run a short training session to verify everything works end-to-end.
For full training, use the CLI: `python -m src.train --config configs/default.yaml`

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from torch.cuda.amp import GradScaler

# Setup training components
criterion = build_loss(
    name=config.training.loss.name,
    bce_weight=config.training.loss.bce_weight,
    dice_weight=config.training.loss.dice_weight,
)
optimizer = AdamW(model.parameters(), lr=config.training.optimizer.lr, weight_decay=config.training.optimizer.weight_decay)
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=config.training.scheduler.T_0)
scaler = GradScaler(enabled=config.training.mixed_precision and device.type == 'cuda')

# Training loop
train_losses = []
val_losses = []
val_f1s = []

for epoch in range(config.training.epochs):
    train_loss = train_one_epoch(
        model, loaders['train'], criterion, optimizer, scaler,
        device, epoch, config,
    )
    val_loss, metrics = validate(
        model, loaders.get('val', loaders['train']),
        criterion, device, config,
    )
    scheduler.step()
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_f1s.append(metrics['f1'])
    
    print(f'Epoch {epoch+1}/{config.training.epochs} | '
          f'Train: {train_loss:.4f} | Val: {val_loss:.4f} | '
          f'F1: {metrics["f1"]:.4f} | IoU: {metrics["iou"]:.4f}')

## Training Curves

In [ ]:
from src.utils import plot_training_curves

fig = plot_training_curves(train_losses, val_losses, val_f1s)
plt.show()

## Quick Validation Check

Visualize a few predictions to sanity-check the model.

In [ ]:
model.eval()
batch = next(iter(loaders.get('val', loaders['train'])))

with torch.no_grad():
    output = model(
        batch['image1'].to(device),
        batch['image2'].to(device),
    )

preds = (torch.sigmoid(output['pred']) > 0.5).cpu().numpy()

# Visualize first 3 samples
for i in range(min(3, batch['image1'].shape[0])):
    fig = visualize_change_detection(
        image_t1=batch['image1'][i].numpy(),
        image_t2=batch['image2'][i].numpy(),
        prediction=preds[i, 0],
        ground_truth=batch['mask'][i, 0].numpy(),
        title=f'Sample {i}',
    )
    plt.show()